# IntiVision Model Training

This notebook documents the complete training pipeline used to build the IntiVision gesture classification model.

The objective is to explain the model architecture, training configuration, preprocessing pipeline, and the reasoning behind each engineering decision.

## Training Pipeline

The training workflow consists of the following stages:

1. Collect the raw gesture dataset.
2. Extract hand regions using MediaPipe.
3. Resize all images to 224 × 224 pixels.
4. Normalize pixel values.
5. Train the CNN model.
6. Save the trained model.
7. Evaluate the model on an independent test dataset.

## Model Selection

Instead of using a large pre-trained architecture, a custom CNN was designed for this project.

The objective was to build a lightweight model capable of real-time inference while keeping the architecture easy to understand, modify, and retrain.

Considering the relatively small dataset and the limited number of gesture classes, a compact CNN provides sufficient capacity without unnecessary computational complexity.

## Model Architecture

### CNN Architecture

The IntiVision model uses a lightweight Convolutional Neural Network (CNN) designed specifically for real-time hand gesture recognition.

The architecture gradually extracts visual features through multiple convolutional blocks before performing gesture classification using fully connected layers.

In [2]:
import tensorflow as tf

from pathlib import Path

In [3]:
PROJECT_ROOT = Path.cwd().parent

MODEL_PATH = PROJECT_ROOT / "models" / "intivision_v2_2.keras"

model = tf.keras.models.load_model(MODEL_PATH)

In [4]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 86528)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │    11,075,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 5)              │           645 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 33,508,817 (127.83 MB)

 Trainable params: 11,169,605 (42.61 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 22,339,212 (85.22 MB)

### Model Summary Analysis

The model contains approximately **11.2 million trainable parameters**, while the optimizer maintains additional internal parameters during training.

The convolution blocks progressively extract visual features, while the fully connected layers perform the final gesture classification.

### Feature Extraction

The network contains three convolutional blocks.

Each block consists of:

- a convolution layer for feature extraction,
- followed by a max pooling layer for spatial downsampling.

As the network becomes deeper, it learns increasingly complex visual representations of the hand gestures.

### Convolution Filter Progression

The number of convolution filters increases from **32** to **64**, and finally **128**.

This progressive increase allows the network to learn increasingly complex visual features while keeping the model lightweight enough for real-time inference.

Given the relatively small number of gesture classes and the dataset size, a three-block CNN architecture provides a good balance between model capacity and computational efficiency.

### Max Pooling

Each convolution layer is followed by a max pooling operation.

Max pooling reduces the spatial dimensions of the feature maps, decreasing computational cost while preserving the most informative visual features.

This also helps the model become more robust to small variations in hand position.

### Dropout

A dropout layer is applied before the final classification layer.

Dropout randomly deactivates a portion of neurons during training, helping reduce overfitting and improving the model's ability to generalize to unseen hand gestures.

### Output Layer

The final dense layer contains five neurons with a Softmax activation function.

Each neuron represents one gesture class:

- Stop
- Safe
- Not Safe
- Emergency
- Help Code

The Softmax function converts the network outputs into class probabilities, allowing the model to select the most likely gesture.

### Input Image Size

All hand images are resized to **224 × 224** pixels before being passed to the neural network.

Using a fixed input resolution ensures a consistent input format for the CNN while preserving sufficient visual detail for gesture recognition.

The selected resolution provides a practical balance between computational efficiency and classification performance for real-time applications.

### Flatten Layer

After the final convolution block, the extracted feature maps are flattened into a one-dimensional feature vector.

This transformation allows the fully connected layers to use the learned visual features for final gesture classification.

## Training Data Pipeline

The MediaPipe-processed dataset is used for model training.

The dataset is split into training and validation subsets, and all images are resized and normalized before being passed to the CNN.

In [6]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
TRAINING_DATASET_DIR = PROJECT_ROOT / "dataset_mediapipe"

TRAINING_DATASET_DIR

PosixPath('/Users/serhaterbil/Desktop/intivision/ai-service/dataset_mediapipe')

In [7]:
import tensorflow as tf

IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
VALIDATION_SPLIT = 0.20
RANDOM_SEED = 42

In [8]:
train_dataset = tf.keras.utils.image_dataset_from_directory(
    TRAINING_DATASET_DIR,
    validation_split=VALIDATION_SPLIT,
    subset="training",
    seed=RANDOM_SEED,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
)

validation_dataset = tf.keras.utils.image_dataset_from_directory(
    TRAINING_DATASET_DIR,
    validation_split=VALIDATION_SPLIT,
    subset="validation",
    seed=RANDOM_SEED,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
)

Found 2745 files belonging to 5 classes.
Using 2196 files for training.
Found 2745 files belonging to 5 classes.
Using 549 files for validation.


### Train–Validation Split

The dataset was divided into:

- **2196 training images**
- **549 validation images**

This corresponds to an **80/20 split**.

The training subset is used to optimize the model parameters, while the validation subset is used to monitor generalization performance during training.

In [9]:
normalization_layer = tf.keras.layers.Rescaling(1.0 / 255)

In [10]:
train_dataset = train_dataset.map(
    lambda images, labels: (
        normalization_layer(images),
        labels,
    )
)

validation_dataset = validation_dataset.map(
    lambda images, labels: (
        normalization_layer(images),
        labels,
    )
)

In [12]:
AUTOTUNE = tf.data.AUTOTUNE

train_dataset = train_dataset.prefetch(
    buffer_size=AUTOTUNE
)

validation_dataset = validation_dataset.prefetch(
    buffer_size=AUTOTUNE
)

## Model Compilation

The model is compiled using:

- **Adam optimizer**
- **Sparse categorical crossentropy loss**
- **Accuracy metric**

These choices are suitable for multi-class classification with integer-encoded labels.

In [13]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

In [14]:
print("Optimizer:", model.optimizer.__class__.__name__)
print("Loss:", model.loss)
print("Metrics:", model.metrics_names)

Optimizer: Adam
Loss: sparse_categorical_crossentropy
Metrics: ['loss', 'compile_metrics']


## Training Hyperparameters

| Parameter | Value |
|-----------|-------|
| Image Size | 224 × 224 |
| Batch Size | 32 |
| Epochs | 20 |
| Optimizer | Adam |
| Loss Function | Sparse Categorical Crossentropy |
| Validation Split | 20% |
| Number of Classes | 5 |

## Training Configuration

The IntiVision V2.2 model was trained for **20 epochs** using the MediaPipe-processed dataset.

The training process used:

- 2196 training images
- 549 validation images
- Adam optimizer
- Sparse categorical crossentropy loss
- Accuracy as the primary training metric

The model was not retrained inside this notebook. Instead, this notebook documents the configuration used during the original training process.

In [15]:
TRAINING_EPOCHS = 20

print("Training epochs:", TRAINING_EPOCHS)
print("Training images:", 2196)
print("Validation images:", 549)

Training epochs: 20
Training images: 2196
Validation images: 549


### Original Training Step

The original training script used the following structure:

```python
model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=20,
)

# Conclusion

This notebook documents the complete training workflow used to develop the IntiVision V2.2 gesture classification model.

The custom CNN architecture, preprocessing pipeline, training configuration, and optimization strategy were designed to achieve reliable real-time gesture recognition while maintaining computational efficiency.

The performance of the trained model is analyzed in the next notebook.